In [0]:
class Silver_circuits():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    silver_path = "formula1_race_project/silver"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        read_df=spark.read.table('formula1_race.bronze.circuits')
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)

        colrename_df= (colrename_df.withColumnRenamed('lat','latitude')
                            .withColumnRenamed('lng','longitude')
                            .withColumnRenamed('alt','altitude')
                            .withColumnRenamed('name','circuit_name')
                            .withColumnRenamed('country','circuit_country')
                            .withColumnRenamed('location','circuit_location')
        )
        from pyspark.sql.functions import expr
        colrename_df=(colrename_df.withColumn('circuit_country',
                                        expr("case when circuit_country = 'USA' then 'United States'"
                                               "when circuit_country = 'UK' then 'United Kingdom'"
                                               "when circuit_country = 'UAE' then 'United Arab Emirates'"
                                               "else  circuit_country end as circuit_country")
                                            )
                       )
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col
        apply_tran_df= (colrename_df.selectExpr("circuit_id","circuit_ref","circuit_name",
                                              "circuit_location","circuit_country","latitude",
                                              "longitude","altitude","circuits_ingestion_date","source")
                        )
        return apply_tran_df
    def write_output(self,apply_tran_df):
        apply_tran_df.write.mode("overwrite").saveAsTable("formula1_race.silver.circuits")
        display(spark.sql("select count(*) from formula1_race.silver.circuits"))
        print("Data write into sliver circuits table is Done")
        
    
       
        
    def process(self):
        print("Started silver-ingestion-circuits  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
       
    


In [0]:
Silver_circuits_instance = Silver_circuits("circuits")
Silver_circuits_instance.process()